In [ ]:
"""Get all imports."""
import csv
import gc
import json
import time

import httpx
from bmt import Toolkit

from gandalf import (
    CSRGraph,
    build_graph_from_jsonl,
    compare_trapi_messages,
    debug_missing_edge,
    enrich_knowledge_graph,
    lookup,
    validate_trapi_response,
)
from gandalf.search import (
    do_one_hop,
)

t0 = time.perf_counter()
bmt = Toolkit()
t1 = time.perf_counter()
print(f"BMT Initialization took: {t1 - t0} seconds.")


In [ ]:
"""Build a new graph from jsonl files. Saves to pickle."""

# Build graph from edges and nodes
graph = build_graph_from_jsonl(
    edge_jsonl_path="../02_20_2026/edges.jsonl",
    node_jsonl_path="../02_20_2026/nodes.jsonl",
)

# Save for fast reloading
graph.save_mmap("../02_20_2026/gandalf_mmap")
print("Graph saved!")

In [ ]:
"""Load in a pickled graph."""

# Load saved graph (takes ~1-2 seconds instead of 280!)
graph = CSRGraph.load_mmap("../02_20_2026/gandalf_mmap")

# Freeze all objects allocated so far (graph + BMT) into a permanent
# generation that the cyclic GC will never scan.  This makes Gen 2
# collections cheap because they skip the large CSR arrays.
gc.collect()
gc.freeze()
# Raise thresholds so Gen 2 collections are less frequent even for
# the (now-small) unfrozen query-time object set.
gc.set_threshold(50_000, 50, 50)

In [ ]:
CSRGraph.save_mmap(directory="../12_17_2025/gandalf_mmap")

In [ ]:
"""Just set the start and end nodes you're looking for."""
start_node = "CHEBI:45783"  # Imatinib
end_node = "MONDO:0004979"  # Asthma
start_idx = graph.get_node_idx(start_node)
end_idx = graph.get_node_idx(end_node)

start_idx = graph.get_node_idx(start_node)
end_idx = graph.get_node_idx(end_node)

print(f"Start degree: {graph.degree(start_idx):,}")
print(f"End degree: {len(graph.incoming_neighbors(end_idx)):,}")

In [ ]:
valid_predicates = []
for predicate in ["biolink:treats",
        "biolink:affects",
        "biolink:regulates",
        "biolink:increases_expression_of",
        "biolink:decreases_expression_of",
        "biolink:gene_associated_with_condition",
        "biolink:has_metabolite",
        "biolink:metabolized_by",
        "biolink:applied_to_treat",
        "biolink:contraindicated_for",
        "biolink:directly_physically_interacts_with",
        "biolink:has_contraindication",
        "biolink:subject_of_treatment_application_or_study_for_treatment_by",
        "biolink:contribution_from"]:
    if bmt.get_element(predicate) is not None:
        valid_predicates.append(predicate)

print(valid_predicates)

In [ ]:
"""Do Pathfinder Imatinib to Asthma."""

t0 = time.perf_counter()

pathfinder_results = lookup(graph, {
  "message": {
    "query_graph": {
      "nodes": {
        "n0": {
          "ids": ["CHEBI:45783"]
        },
        "n1": {},
        "n2": {},
        "n3": {
          "ids": ["MONDO:0004979"]
        }
      },
      "edges": {
        "e0": {
          "subject": "n0",
          "object": "n1",
          "predicates": [
                "biolink:contributes_to",
                "biolink:contribution_from",
                "biolink:affects",
                "biolink:affected_by",
                "biolink:interacts_with",
                "biolink:acts_upstream_of",
                "biolink:has_upstream_actor",
                "biolink:enables",
                "biolink:enabled_by",
                "biolink:produces",
                "biolink:produced_by",
                "biolink:has_participant",
                "biolink:participates_in",
                "biolink:derives_from",
                "biolink:derives_into",
                "biolink:transcribed_to",
                "biolink:transcribed_from",
                "biolink:translates_to",
                "biolink:translation_of",
                "biolink:has_gene_product",
                "biolink:gene_product_of"
            ]
        },
        "e1": {
          "subject": "n1",
          "object": "n2",
          "predicates": [
                "biolink:contributes_to",
                "biolink:contribution_from",
                "biolink:affects",
                "biolink:affected_by",
                "biolink:interacts_with",
                "biolink:acts_upstream_of",
                "biolink:has_upstream_actor",
                "biolink:enables",
                "biolink:enabled_by",
                "biolink:produces",
                "biolink:produced_by",
                "biolink:has_participant",
                "biolink:participates_in",
                "biolink:derives_from",
                "biolink:derives_into",
                "biolink:transcribed_to",
                "biolink:transcribed_from",
                "biolink:translates_to",
                "biolink:translation_of",
                "biolink:has_gene_product",
                "biolink:gene_product_of"
            ]
        },
        "e2": {
          "subject": "n2",
          "object": "n3",
          "predicates": [
                "biolink:contributes_to",
                "biolink:contribution_from",
                "biolink:affects",
                "biolink:affected_by",
                "biolink:interacts_with",
                "biolink:acts_upstream_of",
                "biolink:has_upstream_actor",
                "biolink:enables",
                "biolink:enabled_by",
                "biolink:produces",
                "biolink:produced_by",
                "biolink:has_participant",
                "biolink:participates_in",
                "biolink:derives_from",
                "biolink:derives_into",
                "biolink:transcribed_to",
                "biolink:transcribed_from",
                "biolink:translates_to",
                "biolink:translation_of",
                "biolink:has_gene_product",
                "biolink:gene_product_of"
            ]
        }
      }
    }
  }
}, bmt)

t1 = time.perf_counter()
print(f"Pathfinder returned {len(pathfinder_results['message']['results'])} results in {t1 - t0}s")


def is_result_connected(result: dict, knowledge_graph: dict) -> bool:
    """
    Check if a TRAPI result is fully connected.

    Traverses the edges bound in the result and verifies that all bound nodes
    are reachable within a single connected component.

    Args:
        result: A single TRAPI result containing node_bindings and edge_bindings.
        knowledge_graph: The TRAPI knowledge_graph with 'nodes' and 'edges'.

    Returns:
        True if all nodes in the result are connected via the result's edges.
    """
    # Collect all node IDs referenced in this result's node_bindings
    bound_nodes = set()
    for qnode_id, bindings in result.get("node_bindings", {}).items():
        for binding in bindings:
            bound_nodes.add(binding["id"])

    if len(bound_nodes) <= 1:
        return True

    # Collect all edge IDs referenced in this result's edge_bindings
    # and build an adjacency list from those edges
    adjacency: dict[str, set[str]] = {node: set() for node in bound_nodes}

    for qedge_id, bindings in result.get("analyses", [{}])[0].get("edge_bindings", {}).items():
        for binding in bindings:
            edge_id = binding["id"]
            edge = knowledge_graph.get("edges", {}).get(edge_id)
            if edge is None:
                continue
            subj = edge["subject"]
            obj = edge["object"]
            # Only consider edges where both endpoints are in our bound nodes
            if subj in bound_nodes and obj in bound_nodes:
                adjacency[subj].add(obj)
                adjacency[obj].add(subj)

    # BFS from an arbitrary bound node
    start = next(iter(bound_nodes))
    visited = set()
    queue = [start]
    visited.add(start)

    while queue:
        current = queue.pop(0)
        for neighbor in adjacency.get(current, set()):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    return visited == bound_nodes

for result in pathfinder_results["message"]["results"]:
    if not is_result_connected(result, pathfinder_results["message"]["knowledge_graph"]):
        print("Found non-connected result!")

# result = validate_trapi_response(graph, pathfinder_results)
# print(result.summary())

In [ ]:
result = debug_missing_edge(graph, "NCBIGene:596", "biolink:affects", "CHEBI:30742")
print(result)

# print(diagnose_graph_edge_storage(graph, "GO:0022408"))

In [ ]:
"""Check all edges for number of neighbors."""

line_count = 0
edges = []
edge_ids = set()

with open("../02_20_2026/edges.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line_count % 1_000_000 == 0:
            print(f"  Processed {line_count:,} edges...")

        # if line_count < 26_000_000:
        #     line_count += 1
        #     continue

        data = json.loads(line)

        if data["id"] in edge_ids:
            print("found duplicate edge id!")
            break
        else:
            edge_ids.add(data["id"])
        # if (
        #     data["predicate"] == "biolink:affects" and
        #     (
        #         data["subject"] == "NCBIGene:2277" or
        #         data["object"] == "NCBIGene:2277"
        #     ) and
        #     (
        #         data["object"] == "CHEBI:30742" or
        #         data["subject"] == "CHEBI:30742"
        #     )
        # ):
        #     edges.append(data)
        # if line_count % 1000 == 0:
        #     edges.append(data)

        # if (
        #     "qualifiers" in data
        # ):
        #     edges.append(data)

        # if data["predicate"] == "biolink:genetic_association":
        #     if (
        #         "biolink:Gene" in nodes[data["subject"]]["category"] and
        #         "biolink:DiseaseOrPhenotypicFeature" in nodes[data["object"]]["category"]
        #     ):
        #         print(data)
        #         break
        # if line_count % 10_000 == 0:
        #     edges.append(data)
        # if data["subject"] == "MONDO:0007186" and data["object"] == "MONDO:0007186":
        #     neighbors += 1
        #     edges.append(data)
        # if data["subject"] == "MONDO:0004979" or data["object"] == "MONDO:0004979":
        #     neighbors += 1
        line_count += 1
        # if len(edges) > 1000:
        #     break

with open("./sample_edges.json", "w", encoding="utf-8") as f:
    json.dump(edges, f, indent=2)

print(len(edge_ids))


In [ ]:
lookup_paths = lookup(graph, {
  "message": {
    "query_graph": {
      "nodes": {
        "SN": { "categories": ["biolink:ChemicalEntity"] },
        "ON": {
          "ids": ["MONDO:0007186"],
          "categories": ["biolink:DiseaseOrPhenotypicFeature"]
        },
        "e": { "categories": ["biolink:ChemicalEntity"] },
        "i": { "categories": ["biolink:BiologicalEntity"] }
      },
      "edges": {
        "edge_0": {
          "subject": "e",
          "object": "ON",
          "predicates": ["biolink:treats"]
        },
        "edge_1": {
          "subject": "i",
          "object": "SN",
          "predicates": ["biolink:affects"],
          "qualifier_constraints": [
            {
              "qualifier_set": [
                {
                  "qualifier_type_id": "biolink:object_aspect_qualifier",
                  "qualifier_value": "activity_or_abundance"
                },
                {
                  "qualifier_type_id": "biolink:object_direction_qualifier",
                  "qualifier_value": "decreased"
                }
              ]
            }
          ]
        },
        "edge_2": {
          "subject": "i",
          "object": "e",
          "predicates": ["biolink:affects"],
          "qualifier_constraints": [
            {
              "qualifier_set": [
                {
                  "qualifier_type_id": "biolink:object_aspect_qualifier",
                  "qualifier_value": "activity_or_abundance"
                },
                {
                  "qualifier_type_id": "biolink:object_direction_qualifier",
                  "qualifier_value": "decreased"
                }
              ]
            }
          ]
        }
      }
    }
  }
}, bmt)

with open("../12_17_2025/heartburn_response.json", "w") as f:
    json.dump(lookup_paths, f, indent=2)

In [ ]:
import httpx

with open("./expanded_messages.json", "r", encoding="utf-8") as f:
    messages = json.load(f)

all_results = 0
for indx, message in enumerate(messages):
    message["parameters"]["tiers"] = [1]
    t0 = time.perf_counter()
    try:
        with httpx.Client(timeout=60) as client:
            print(message)
            response = client.post(
                "https://automat.renci.org/translatorkg/query",
                # "https://automat.renci.org/translatorkg-memgraph/query?subclass=true",
                json=message,
            )
            response.raise_for_status()
            response = response.json()
            # with open(f"heartburn_responses/response_neo4j_{indx}.json", "w", encoding="utf-8") as f:
            #     json.dump(response, f, indent=2)
            num_results = len((response.get("message") or {}).get("results") or [])
            result = validate_trapi_response(graph, response)
            print(result.summary())
    except httpx.ReadTimeout:
        print("Timed out")
        num_results = 0
    except httpx.HTTPError as e:
        print("Got bad response:", response.content)
        num_results = 0
    print(f"Returned {num_results} results")
    all_results += num_results
    t1 = time.perf_counter()
    print(f"Query took {t1 - t0} seconds")

print("Total results:", all_results)

In [ ]:
with open("./expanded_messages.json", "r", encoding="utf-8") as f:
    messages = json.load(f)

all_results = 0
for indx, message in enumerate(messages):
    t0 = time.perf_counter()
    print(message)
    response = lookup(graph, message, bmt, subclass=False)
    # with open(f"heartburn_responses/response_gandalf_{indx}.json", "w", encoding="utf-8") as f:
    #     json.dump(response, f, indent=2)
    all_results += len(response["message"]["results"])
    # if len(response["message"]["results"]) > 0:
    #     with open("lookup_paths.json", "w", encoding="utf-8") as f:
    #         json.dump(response, f, indent=2)
    result = validate_trapi_response(graph, response)
    print(result.summary())
    t1 = time.perf_counter()
    print(f"Query took {t1 - t0} seconds")

print("Total results:", all_results)

In [ ]:
def convert_path_to_sentence(source, target, path, knowledge_graph, logger):
    path_node_list = [source]
    next_node = source
    while target not in path_node_list:
        for edge_id in path:
            if knowledge_graph["edges"][edge_id]["subject"] == path_node_list[-1]:
                if knowledge_graph["edges"][edge_id]["object"] not in path_node_list:
                    path_node_list.append(knowledge_graph["edges"][edge_id]["object"])
            elif knowledge_graph["edges"][edge_id]["object"] == path_node_list[-1]:
                if knowledge_graph["edges"][edge_id]["subject"] not in path_node_list:
                    path_node_list.append(knowledge_graph["edges"][edge_id]["subject"])
    current_node = source
    path_predicate_list = []
    for hop_num, next_node in enumerate(path_node_list[1:]):
        path_predicate_list.append(set())
        for edge_id in path:
            pred = knowledge_graph["edges"][edge_id]["predicate"]
            pred_info = bmt.get_element(pred)
            if not pred_info:
                logger.error("Predicate doesn't exist.")
                continue
            else:
                pred = pred_info.name
            if (
                knowledge_graph["edges"][edge_id]["subject"] == current_node
                and knowledge_graph["edges"][edge_id]["object"] == next_node
            ):
                path_predicate_list[hop_num].add(pred)
            elif (
                knowledge_graph["edges"][edge_id]["object"] == current_node
                and knowledge_graph["edges"][edge_id]["subject"] == next_node
            ):
                inv = bmt.get_inverse_predicate(pred)
                if not inv:
                    logger.error("Predicate doesn't exist.")
                else:
                    path_predicate_list[hop_num].add(pred)
        current_node = next_node
    source_cat = bmt.get_element(knowledge_graph['nodes'][source]['categories'][-1])
    if not source_cat:
        logger.error("Category doesn't exist.")
        raise Exception
    path_sentence = f"{knowledge_graph['nodes'][source]['name']} (a {source_cat.name}) "
    first_hop = True
    for path_node, hop_predicates in zip(path_node_list[1:], path_predicate_list):
        hop_preds = list(hop_predicates)
        if not first_hop:
            path_sentence += ", which "
        else:
            first_hop = False
        if len(hop_preds) == 1:
            path_sentence += f"{hop_preds[0]}"
        else:
            path_sentence += f"either ["
            for hop_pred in hop_preds[:-1]:
                path_sentence += f"{hop_pred} or "
            path_sentence += f"{hop_preds[-1]}]"
        node_cat = bmt.get_element(knowledge_graph['nodes'][path_node]['categories'][-1])
        if not node_cat:
            logger.error("Category doesn't exist.")
            raise Exception
        path_sentence += f" {knowledge_graph['nodes'][path_node]['name']} (a {node_cat.name})"

    print(path_sentence)
    return path_sentence

In [ ]:
payload = {
  "message": {
    "query_graph": {
      "nodes": {
        "SN": { "categories": ["biolink:ChemicalEntity"] },
        "ON": {
          "ids": ["MONDO:0007186"],
          "categories": ["biolink:DiseaseOrPhenotypicFeature"]
        },
        "e": { "categories": ["biolink:ChemicalEntity"] },
        "i": { "categories": ["biolink:BiologicalEntity"] }
      },
      "edges": {
        "edge_0": {
          "subject": "e",
          "object": "ON",
          "predicates": ["biolink:treats"]
        },
        "edge_1": {
          "subject": "i",
          "object": "SN",
          "predicates": ["biolink:affects"],
          "qualifier_constraints": [
            {
              "qualifier_set": [
                {
                  "qualifier_type_id": "biolink:object_aspect_qualifier",
                  "qualifier_value": "activity_or_abundance"
                },
                {
                  "qualifier_type_id": "biolink:object_direction_qualifier",
                  "qualifier_value": "decreased"
                }
              ]
            }
          ]
        },
        "edge_2": {
          "subject": "i",
          "object": "e",
          "predicates": ["biolink:affects"],
          "qualifier_constraints": [
            {
              "qualifier_set": [
                {
                  "qualifier_type_id": "biolink:object_aspect_qualifier",
                  "qualifier_value": "activity_or_abundance"
                },
                {
                  "qualifier_type_id": "biolink:object_direction_qualifier",
                  "qualifier_value": "decreased"
                }
              ]
            }
          ]
        }
      }
    }
  }
}

t0 = time.perf_counter()
with httpx.Client(timeout=600) as client:
    response = client.post(
        "https://automat.renci.org/translatorkg/query?subclass=true",
        # "https://automat.renci.org/translatorkg-memgraph/query?subclass=true",
        json=payload,
    )
    response.raise_for_status()
    response = response.json()

# response = lookup(graph, payload, bmt)

num_results = len((response.get("message") or {}).get("results") or [])
result = validate_trapi_response(graph, response)
print(result.summary())

t1 = time.perf_counter()
print(f"Query took {t1 - t0} seconds")


In [ ]:
t0 = time.perf_counter()
lookup_paths = lookup(graph, {
  "message": {
    "query_graph": {
      "nodes": {
        "SN": { "categories": ["biolink:ChemicalEntity"] },
        "ON": {
          "ids": ["MONDO:0007186"],
          "categories": ["biolink:DiseaseOrPhenotypicFeature"]
        },
        "e": { "categories": ["biolink:ChemicalEntity"] },
        "i": { "categories": ["biolink:BiologicalEntity"] }
      },
      "edges": {
        "edge_0": {
          "subject": "e",
          "object": "ON",
          "predicates": ["biolink:treats"]
        },
        "edge_1": {
          "subject": "i",
          "object": "SN",
          "predicates": ["biolink:affects"],
          "qualifier_constraints": [
            {
              "qualifier_set": [
                {
                  "qualifier_type_id": "biolink:object_aspect_qualifier",
                  "qualifier_value": "activity_or_abundance"
                },
                {
                  "qualifier_type_id": "biolink:object_direction_qualifier",
                  "qualifier_value": "decreased"
                }
              ]
            }
          ]
        },
        "edge_2": {
          "subject": "i",
          "object": "e",
          "predicates": ["biolink:affects"],
          "qualifier_constraints": [
            {
              "qualifier_set": [
                {
                  "qualifier_type_id": "biolink:object_aspect_qualifier",
                  "qualifier_value": "activity_or_abundance"
                },
                {
                  "qualifier_type_id": "biolink:object_direction_qualifier",
                  "qualifier_value": "decreased"
                }
              ]
            }
          ]
        }
      }
    }
  }
}, bmt)

t1 = time.perf_counter()
print(f"Query took {t1 - t0} seconds")

with open("lookup_paths.json", "w", encoding="utf-8") as f:
    json.dump(lookup_paths, f, indent=2)

# result = validate_trapi_response(graph, lookup_paths)
# print(result.summary())

In [ ]:
asprin_to_colorectral_cancer = lookup(graph, {
  "message": {
    "query_graph": {
      "nodes": {
        "n0": {
          "ids": ["CHEBI:15365"]
        },
        "n1": {},
        "n2": {},
        "n3": {
          "ids": ["MONDO:0005575"]
        }
      },
      "edges": {
        "e0": {
          "subject": "n0",
          "object": "n1",
          "predicates": ["biolink:related_to"]
        },
        "e1": {
          "subject": "n1",
          "object": "n2",
          "predicates": ["biolink:treats", "biolink:affects", "biolink:regulates", "biolink:gene_associated_with_condition", "biolink:has_metabolite", "biolink:applied_to_treat", "biolink:directly_physically_interacts_with", "biolink:has_contraindication", "biolink:subject_of_treatment_application_or_study_for_treatment_by", "biolink:contribution_from"]
        },
        "e2": {
          "subject": "n2",
          "object": "n3",
          "predicates": ["biolink:related_to"]
        }
      }
    }
  }
}, bmt)

In [ ]:
"""Do small query."""
message = {
  "message": {
    "query_graph": {
      "nodes": {
        "n0": { "ids": ["CHEBI:45783"] },
        "n1": {},
        "n2": { "ids": ["MONDO:0004979"] }
      },
      "edges": {
        "e0": {
          "subject": "n0",
          "object": "n1",
          "predicates": ["biolink:related_to"]
        },
        "e1": {
          "subject": "n1",
          "object": "n2",
          "predicates": ["biolink:related_to"]
        }
      }
    }
  }
}

t0 = time.perf_counter()

automat_results = {}
try:
    with httpx.Client(timeout=60) as client:
        response = client.post(
            "https://automat.renci.org/translatorkg/query",
            json=message,
        )
        response.raise_for_status()
        automat_results = response.json()
        num_results = len((automat_results.get("message") or {}).get("results") or [])
        result = validate_trapi_response(graph, automat_results)
        print(result.summary())
except httpx.ReadTimeout:
    print("Timed out")
    num_results = 0
except httpx.HTTPError as e:
    print("Got bad response:", response.content)
    num_results = 0

t1 = time.perf_counter()
print(f"Automat returned {len(automat_results['message']['results'])} results in {t1 - t0}s")

with open("automat_heartburn_response.json", "w", encoding="utf-8") as f:
    json.dump(automat_results, f, indent=2)

t0 = time.perf_counter()
gandalf_results = lookup(graph, message, bmt, subclass=True)

t1 = time.perf_counter()
print(f"Gandalf returned {len(gandalf_results['message']['results'])} results in {t1 - t0}s")

with open("gandalf_heartburn_response.json", "w", encoding="utf-8") as f:
    json.dump(gandalf_results, f, indent=2)

missing_results = compare_trapi_messages(automat_results["message"], gandalf_results["message"])

for missing_result in missing_results:
    print(missing_result)

# result = validate_trapi_response(graph, pathfinder_results)
# print(result.summary())

In [ ]:
with open("/Users/mwang/CoVar/translator/biopack/shepherd/scripts/aragorn/CHEBI_45783_MONDO_0004979_response.json", "r", encoding="utf-8") as f:
    message = json.load(f)

hydrated_response = enrich_knowledge_graph(message, graph)

with open("./hydrated_response.json", "w", encoding="utf-8") as f:
    json.dump(hydrated_response, f, indent=2)
